In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons, save_treatment_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"G:\Calcium Imaging\GCaMP8s_EX369\NBDL-MQNMFA Combo")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── NBDL-MQNMFA Combo
    ├── Week 2
    │   ├── 2-1
    │   ├── 2-1_LOW_25m_3m
    │   ├── 2-1_LOW_2m
    │   ├── 2-2
    │   ├── 2-2_LOW_31m_9m
    │   ├── 2-2_LOW_8m
    │   ├── 2-3
    │   ├── 2-3_LOW_14m
    │   ├── 2-3_LOW_37m_15m
    │   ├── 2-4
    │   ├── 2-4_HIGH_24m_3m
    │   ├── 2-4_HIGH_2m
    │   ├── 2-5
    │   ├── 2-5_HIGH_30m_10m
    │   ├── 2-5_HIGH_8m
    │   ├── 2-6
    │   ├── 2-6_HIGH_15m
    │   ├── 2-6_HIGH_37m_17m
    │   └── metrics
    └── metrics


In [5]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 2-1
  Concatenated mode: 3 section(s) (baseline, treatment_1, treatment_2)
  Traces: 466 ROIs, 10950 frames @ 15.0 Hz
  ROI filter: 402/466 kept (86.3%)
  Spikes: 22231/119569 kept | neurons 402 -> 402
  Grouping (combined): | combined=24
  Section comparison (combined, treatment_1): 24 groups | mean delta corr=-0.201 | 15 surviving sub-groups
  Section comparison (combined, treatment_2): 24 groups | mean delta corr=-0.471 | 5 surviving sub-groups

 Processing: 2-2
  Concatenated mode: 3 section(s) (baseline, treatment_1, treatment_2)
  Traces: 411 ROIs, 10950 frames @ 15.0 Hz
  ROI filter: 355/411 kept (86.4%)
  Spikes: 16512/107650 kept | neurons 355 -> 355
  Grouping (combined): | combined=16
  Section comparison (combined, treatment_1): 16 groups | mean delta corr=-0.345 | 5 surviving sub-groups
  Section comparison (combined, treatment_2): 16 groups | mean delta corr=-0.407 | 3 surviving sub-groups

 Processing: 2-3
  Concatenated mode: 3 section(s) (baseline, treatm

In [6]:
 
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: G:\Calcium Imaging\GCaMP8s_EX369\NBDL-MQNMFA Combo\Week 2
child  n_videos  n_neurons  n_groups_combined  mean_group_size_combined  median_group_size_combined  mean_group_corr_combined  mean_spikes_per_group_combined  frac_grouped  frac_ungrouped  decay_tau_seconds_mean_unweighted  half_max_width_seconds_mean_unweighted  rise_slope_hz_mean_unweighted  decay_tau_seconds_mean_weighted  half_max_width_seconds_mean_weighted  rise_slope_hz_mean_weighted  decay_tau_seconds_mean_grouped  half_max_width_seconds_mean_grouped  rise_slope_hz_mean_grouped  decay_tau_seconds_mean_ungrouped  half_max_width_seconds_mean_ungrouped  rise_slope_hz_mean_ungrouped  spike_frequency_mean_unweighted  spike_frequency_mean_weighted  spike_frequency_mean_grouped  spike_frequency_mean_ungrouped  decay_tau_seconds_var_unweighted  decay_tau_seconds_within_unweighted  decay_tau_seconds_between_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds_b

In [7]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)

save_treatment_comparisons(tree)